In [106]:
# !pip install transformers sentence-transformers faiss-cpu gradio pymupdf

In [107]:
#!pip install -U transformers

In [108]:
#!pip install groq

In [ ]:
from groq import Groq

client = Groq(api_key="Your_API_KEY")   #Creates Groq client to call LLM API

In [110]:
import fitz  ##read PDFs
import faiss     ## vector Search
import numpy as np   ## arrays
import gradio as gr   ## UI
from sentence_transformers import SentenceTransformer ## Used for converting text → embeddings (vectors)
from transformers import pipeline

In [111]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [112]:
embed_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')  #Loads embedding model
                                                                              #Converts text into numerical vectors for similarity search

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [113]:
chunks = []   ##all text pieces
index = None   ## FAISS index
metadata = []
chat_history = []
latest_answer = ""

In [114]:
def load_pdf(file):
    doc = fitz.open(file)  ## Opens PDF file
    #Extracts text from every page
    text = ""
    for page in doc:
        text += page.get_text()
    #Returns full PDF text
    return text

In [115]:
def chunk_text(text, chunk_size=700, overlap=150):       ##Breaks text into smaller pieces and Overlap ensures context continuity
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):       ## Sliding window chunking
        chunks.append(text[i:i+chunk_size])
    return chunks

In [116]:
def create_index(chunks):
    embeddings = embed_model.encode(chunks)      ##Converts each chunk → vector
    dim = embeddings.shape[1]             ## Gets vector dimension

    index = faiss.IndexFlatL2(dim)     ##Creates FAISS index (L2 distance search)
    index.add(np.array(embeddings))    ## Stores embeddings in FAISS

    return index

In [117]:
def search(query, index, chunks, k=12):
    query_vec = embed_model.encode([query])
    distances, indices = index.search(np.array(query_vec), k)

    results = []
    scores = []

    for i, idx in enumerate(indices[0]):
        results.append(chunks[idx])
        scores.append(distances[0][i])

    return results, scores, indices[0]

In [118]:
import re

# -----------------------------
# Clean Retrieved Chunks
# -----------------------------
def clean_chunks(chunks):
    cleaned = []
    for chunk in chunks:
        # remove MCQ options (A. B. C. D.)
        chunk = re.sub(r'\b[A-D]\.\s.*', '', chunk)   ## Removes MCQ options (A, B, C, D)

        # remove extra newlines
        chunk = re.sub(r'\n+', '\n', chunk)   ## Removes extra newlines

        cleaned.append(chunk.strip())

    return cleaned

In [119]:
def filter_chunks(chunks, query):              ## Keyword-based filtering (improves relevance)
    query_words = set(query.lower().split())   ## Break query into words

    scored = []
    for chunk in chunks:
        chunk_words = set(chunk.lower().split())
        score = len(query_words & chunk_words)     ## Measures overlap between query and chunk
        scored.append((score, chunk))

    scored.sort(reverse=True)     ## sorts best chunks first

    return [c for s, c in scored if s > 0][:8]

In [120]:
def generate_answer(query, context):
    try:
      ## Creates prompt for LLM & Forces model to use only retrieved data
        prompt = f"""
Answer the question using ONLY the context.

Context:
{context}

Question: {query}

Answer:
"""

        response = client.chat.completions.create(       ## Calls Groq LLM
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],     ##Sends prompt as chat message
            temperature=0.2                             ## Low randomness → more accurate answers
        )

        return response.choices[0].message.content.strip()      ## Extracts final answer

    except Exception as e:
        import traceback
        traceback.print_exc()   # 🔥 FULL ERROR
        return f"❌ API ERROR: {str(e)}"

In [121]:
import re

def highlight_answer_text(answer):
    # Extract keywords (ignore small words)
    words = answer.split()
    keywords = [w for w in words if len(w) > 4]

    # remove duplicates
    keywords = list(set(keywords))

    highlighted = answer

    for word in keywords[:8]:  # limit keywords
        pattern = re.compile(rf"\b({re.escape(word)})\b", re.IGNORECASE)
        highlighted = pattern.sub(r"<b>\1</b>", highlighted)

    return highlighted

In [122]:
def export_answer(answer):
    with open("answer.txt", "w") as f:
        f.write(answer)
    return "answer.txt"

In [123]:
def process_pdfs(files):
    global chunks, index, metadata

    all_chunks = []
    metadata = []   # reset metadata

    for file in files:
        text = load_pdf(file.name)

        file_chunks = chunk_text(text)

        for chunk in file_chunks:
            all_chunks.append(chunk)

            # store metadata for each chunk
            metadata.append({
                "source": file.name
            })

    chunks = all_chunks

    # create FAISS index
    index = create_index(chunks)

    return f"✅ Processed {len(files)} files with {len(chunks)} chunks"

In [124]:
def save_latest_answer(answer):
    global latest_answer
    latest_answer = answer

In [125]:
def ask_question(query):
    global index, chunks, metadata, chat_history

    if index is None:
        return "⚠️ Please upload and process documents first."

    # 🔹 Retrieve
    relevant_chunks, scores, indices = search(query, index, chunks)

    # 🔹 Clean + Filter
    relevant_chunks = clean_chunks(relevant_chunks)
    relevant_chunks = filter_chunks(relevant_chunks, query)

    # 🔹 Context
    context = "\n".join(relevant_chunks[:6])

    # 🔹 Generate answer
    answer = generate_answer(query, context)

    # ✅ SAVE ANSWER HERE (IMPORTANT)
    save_latest_answer(answer)

    # 🔹 Confidence score
    confidence = round(100 - (np.mean(scores[:5]) * 10), 2)

    # 🔹 Sources
    sources = set()
    for idx in indices[:5]:
        sources.add(metadata[idx]["source"])

    sources_text = "\n".join([f"- {s}" for s in sources])

    # 🔹 Highlight
    highlighted_answer = highlight_answer_text(answer)

    # 🔹 Chat memory
    chat_history.append((query, answer))

    # 🔹 Final output
    return f"""
💡 Answer:
{answer}

📊 Confidence: {confidence}%

📄 Sources:
{sources_text}

🔍 Highlighted Answer:
{highlighted_answer}
"""

In [126]:
import tempfile

def export_answer_file():
    global latest_answer

    if latest_answer == "":
        return None

    # create temp file
    file_path = tempfile.NamedTemporaryFile(delete=False, suffix=".txt").name

    with open(file_path, "w") as f:
        f.write(latest_answer)

    return file_path

In [127]:
with gr.Blocks() as demo:      ## Creates UI
    gr.Markdown("# 🤖 Smart Document Assistant (Advanced RAG)")

    file_input = gr.File(file_count="multiple", label="Upload PDF(s)")    # Upload PDFs
    process_btn = gr.Button("Process Documents")
    status = gr.Textbox(label="Status")

    chatbot = gr.Chatbot()            ## Chat UI
    msg = gr.Textbox(label="Ask something")

    # ==============================
    # ✅ NEW: Export Button UI
    # ==============================
    export_btn = gr.Button("⬇ Download Answer")
    file_output = gr.File(label="Download your answer")

    def respond(message, chat_history_ui):      ## Handles user input
        reply = ask_question(message)
        chat_history_ui.append((message, reply))
        return "", chat_history_ui

    process_btn.click(process_pdfs, inputs=file_input, outputs=status)    ## Button action
    msg.submit(respond, [msg, chatbot], [msg, chatbot])    ## Send message

    # ==============================
    # ✅ NEW: Connect export button
    # ==============================
    export_btn.click(fn=export_answer_file, outputs=file_output)

demo.launch()

/tmp/ipykernel_1347/2717598742.py:8: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()            ## Chat UI
/tmp/ipykernel_1347/2717598742.py:8: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()            ## Chat UI


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6c8231f4d1e76298c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
